In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path(r"C:\Users\polok\deep-learning-assignment-1")
DATA_DIR = PROJECT_ROOT / "data"

print("Project exists:", PROJECT_ROOT.exists())
print("Data exists:", DATA_DIR.exists())
print("Train exists:", (DATA_DIR / "train_data.csv").exists())

train = pd.read_csv(DATA_DIR / "train_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Project exists: True
Data exists: True
Train exists: True
Train: (5531451, 190)
Test: (11363762, 190)


In [5]:
print("QUESTION: Can we distinguish train from test?")

numeric_cols = [
    c for c in train.select_dtypes(include=np.number).columns
    if c not in ["customer_ID", "S_2"]
]

N = 300_000

train_sample = train[numeric_cols].sample(
    n=min(N, len(train)),
    random_state=42
)

test_sample = test[numeric_cols].sample(
    n=min(N, len(test)),
    random_state=42
)

train_adv = train_sample.astype(np.float32)
test_adv = test_sample.astype(np.float32)

train_adv["is_test"] = 0
test_adv["is_test"] = 1

adv = pd.concat(
    [train_adv, test_adv],
    ignore_index=True
)

X = adv.drop(columns="is_test")
y = adv["is_test"]

# Free some memory
del train_adv
del test_adv
del train_sample
del test_sample

X_train_adv, X_val_adv, y_train_adv, y_val_adv = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe_classification = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=500,
            solver="liblinear"
        )
    )
])

pipe_classification.fit(X_train_adv, y_train_adv)
proba = pipe_classification.predict_proba(X_val_adv)[:, 1]

auc = roc_auc_score(y_val_adv, proba)

print(f"Adversarial ROC-AUC: {auc:.4f}")

QUESTION: Can we distinguish train from test?


C:\Users\polok\AppData\Local\Temp\ipykernel_2520\684341064.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_adv["is_test"] = 0
C:\Users\polok\AppData\Local\Temp\ipykernel_2520\684341064.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_adv["is_test"] = 1


Adversarial ROC-AUC: 0.7823
